In [ ]:
import requests
import pandas as pd
import time

base_path = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
df = pd.read_csv(base_path + "cell2024_combined.csv")

# Pull down UniProt seq

In [3]:
# Rows with non-null Uniprot IDs
df_with_uniprot = df[df["Uniprot"].notna()].copy()
uniprot_ids = df_with_uniprot["Uniprot"].unique()
print(f"Unique UniProt ID count: {len(uniprot_ids)}")

def fetch_sequence(uid):
    url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            lines = r.text.strip().split("\n")
            return "".join(lines[1:])
        return None
    except:
        return None

sequences = {}
failed = []

for i, uid in enumerate(uniprot_ids):
    seq = fetch_sequence(uid)
    if seq:
        sequences[uid] = seq
    else:
        failed.append(uid)
    if i % 100 == 0:
        print(f"Progress: {i}/{len(uniprot_ids)}, Failed: {len(failed)}")
    time.sleep(0.1)

print(f"\nComplete: Successful {len(sequences)}, Failed {len(failed)}")

seq_df = pd.DataFrame(list(sequences.items()), columns=["Uniprot", "sequence"])
seq_df.to_csv(base_path + "uniprot_seq.csv", index=False)
print("Saved")

Unique UniProt ID count: 896
Progress: 0/896, Failed: 0
Progress: 100/896, Failed: 0
Progress: 200/896, Failed: 0
Progress: 300/896, Failed: 0
Progress: 400/896, Failed: 0
Progress: 500/896, Failed: 0
Progress: 600/896, Failed: 0
Progress: 700/896, Failed: 0
Progress: 800/896, Failed: 0

Complete: Successful 895, Failed 1
Saved


In [19]:
# Rows without

no_uniprot = df[df["Uniprot"].isna()]
genes_to_map = no_uniprot["Gene"].unique()
print(f" Without Uniprot IDs: {len(genes_to_map)} genes")

# Gene name → UniProt ID
def fetch_uniprot_id_by_gene(gene):
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"gene_exact:{gene} AND organism_id:9606 AND reviewed:true",
        "format": "tsv",
        "fields": "accession",
        "size": 1,
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            lines = r.text.strip().split("\n")
            if len(lines) > 1:
                return lines[1].strip()
        return None
    except:
        return None

gene2id = {}
gene_id_failed = []
for i, gene in enumerate(genes_to_map):
    uid = fetch_uniprot_id_by_gene(gene)
    if uid:
        gene2id[gene] = uid
    else:
        gene_id_failed.append(gene)
    if i % 100 == 0:
        print(f"Gene→ID progress: {i}/{len(genes_to_map)}, Failed: {len(gene_id_failed)}")
    time.sleep(0.1)

print(f"\nGene→ID success: {len(gene2id)}, faliure: {len(gene_id_failed)}")

# Fetch
new_ids = [uid for uid in gene2id.values() if uid not in sequences]
print(f"IDs to fetch: {len(new_ids)}")

for i, uid in enumerate(new_ids):
    seq = fetch_sequence(uid)
    if seq:
        sequences[uid] = seq
    else:
        failed.append(uid)
    if i % 100 == 0:
        print(f"Seq progress: {i}/{len(new_ids)}, Failed total: {len(failed)}")
    time.sleep(0.1)

# Fill back UniProt ID
df["Uniprot"] = df["Uniprot"].fillna(df["Gene"].map(gene2id))
print(f"After filling, Uniprot missing: {df['Uniprot'].isna().sum()}")
df.to_csv(base_path + "cell2024_combined_filled.csv", index=False)

seq_df = pd.DataFrame(list(sequences.items()), columns=["Uniprot", "sequence"])
seq_df.to_csv(base_path + "uniprot_seq_2.csv", index=False)

print(f"Saved, All: {len(seq_df)} sequences")

 Without Uniprot IDs: 84 genes
Gene→ID progress: 0/84, Failed: 0

Gene→ID success: 56, faliure: 28
IDs to fetch: 0
After filling, Uniprot missing: 67
Saved, All: 914 sequences


In [16]:
still_na = df[df["Uniprot"].isna()]
print("Missing:", len(still_na))

is_fusion = still_na["Gene"].str.contains("-", na=False)
print("Fusion genes:", is_fusion.sum())
print("Non-fusion genes that failed:", sorted(still_na.loc[~is_fusion, "Gene"].unique()))
print(still_na["source"].value_counts())

Missing: 67
Fusion genes: 67
Non-fusion genes that failed: []
source
main    67
Name: count, dtype: int64


# Generate mutated sequence

In [23]:
# Merge
df = pd.read_csv(base_path + "cell2024_combined_filled.csv")
seq_df = pd.read_csv(base_path + "uniprot_seq_2.csv")

df = df.merge(seq_df, on="Uniprot", how="left")


print(f"With sequences: {df['sequence'].notna().sum()}")
print(f"Without sequences: {df['sequence'].isna().sum()}")

With sequences: 2307
Without sequences: 68


In [ ]:
def parse_variant(variant_str):
    # K222E
    if not isinstance(variant_str, str):
        return None
    import re
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', variant_str.strip())
    if m:
        return m.group(1), int(m.group(2)), m.group(3)
    return None

def make_mutant_sequence(wt_seq, variant_str):
    # Substitute the amino acid at the specified position
    parsed = parse_variant(variant_str)
    if parsed is None:
        return None
    wt_aa, pos, mt_aa = parsed
    if pos < 1 or pos > len(wt_seq):
        return None  # out of bounds
    if wt_seq[pos - 1] != wt_aa:
        return None  # wild-type amino acid does not match
    return wt_seq[:pos - 1] + mt_aa + wt_seq[pos:]

df_seq = df[df['sequence'].notna()].copy()

# Important: only keep rows with a single mutation
df_seq['n_mut'] = df_seq['Mutation'].str.count(r'[A-Z]\d+[A-Z]')
n_before = len(df_seq)
df_seq = df_seq[df_seq['n_mut'] == 1].copy()
df_seq = df_seq.drop(columns=['n_mut'])
print(f"Exclude multi-mutation rows: {n_before - len(df_seq)} → Remaining: {len(df_seq)} (single mutation)")

def build_with_fallback(row):
    # try mutation and alternative mutation; use the one that passes validation
    for mut in [row['Mutation'], row['Mutation (alternative)']]:
        seq = make_mutant_sequence(row['sequence'], mut)
        if seq is not None:
            return pd.Series([seq, mut])
    return pd.Series([None, None])

df_seq[['mutant_sequence', 'Mutation_used']] = df_seq.apply(build_with_fallback, axis=1)

total = len(df_seq)
success = df_seq['mutant_sequence'].notna().sum()
failed_parse = df_seq['mutant_sequence'].isna().sum()

print(f"Success: {success}/{total}")
print(f"Failed rows: {failed_parse}")

failed_rows = df_seq[df_seq['mutant_sequence'].isna()][
    ['Gene', 'Mutation', 'Mutation (alternative)', 'sequence']].head(10)
print(failed_rows)


Exclude multi-mutation rows: 10 → Remaining: 2297 (single mutation)
Success: 2179/2297
Failed rows: 118
       Gene Mutation_final                                           sequence
25     MITF          S298P  MQSESGIVPDFEVGEEFHEEPKTYYELKSQPLKSSSSAEHPGASKP...
67    ABCG8          A632V  MAGKAAEERGLPKGATPQDTSGLQDRLFSSESDNSLYFTYSGQPNT...
117     AGT           L43F  MAPAGVSLRATILCLLAWAGLAAGDRVYIHPFHLVIHNESTCEQLA...
118     AGT          R375Q  MAPAGVSLRATILCLLAWAGLAAGDRVYIHPFHLVIHNESTCEQLA...
119     AGT          T207M  MAPAGVSLRATILCLLAWAGLAAGDRVYIHPFHLVIHNESTCEQLA...
120     AGT          Y281C  MAPAGVSLRATILCLLAWAGLAAGDRVYIHPFHLVIHNESTCEQLA...
124    AHSG          T248M  MKSLVLLLCLAQLWGCHSAPHGPGLIYRQPNCDDPETEEAALVAID...
125    AHSG          T256S  MKSLVLLLCLAQLWGCHSAPHGPGLIYRQPNCDDPETEEAALVAID...
212  ATP2B2          V586M  MGDMTNSDFYSKNQRNESSHGGEFGCTMEELRSLMELRGTEAVVKI...
216  B3GNT3          H328R  MKYLRHRRPNATLILAIGAFTLLLFSLLVSPPTCKVQEQPPAIPEA...


In [30]:
df = df.merge(
    df_seq[["Gene", "Variant", "Mutation_used", "mutant_sequence"]],
    on=["Gene", "Variant"], how="left")

print(f"All rows: {len(df)}")
print(f"Include: (with mutant_sequence): {df['mutant_sequence'].notna().sum()}")
print(f"Exclude rows (no mutant_sequence): {df['mutant_sequence'].isna().sum()}")
print(f"\nDistribution of Mislocalized for rows with mutant_sequence:")
print(df[df['mutant_sequence'].notna()]['Mislocalized'].value_counts())

df.to_csv(base_path + "cell2024_final.csv", index=False)
print(f"\nSaved cell2024_final.csv")

All rows: 2375
Include: (with mutant_sequence): 2179
Exclude rows (no mutant_sequence): 196

Distribution of Mislocalized for rows with mutant_sequence:
Mislocalized
0.0    1943
1.0     236
Name: count, dtype: int64

Saved cell2024_final.csv
